# Exploračná dátová analýza — Australia Weather

**Predmet:** IB031 Úvod do strojového učení  
**Dataset:** Daily Weather Observations – Austrália (2008–2017)  
**Cieľová premenná:** `RainTomorrow` (binárna - bude pršať?) / `RISK_MM` (spojitá - množstvo zrážok)

---

Dataset obsahuje denné meteorologické pozorovania z rôznych lokalít v Austrálii za roky 2008–2017. Dáta pochádzajú od Australian Bureau of Meteorology (BOM). Pre každý deň sú zaznamenané údaje o teplote, zrážkach, vetre, vlhkosti, tlaku, oblačnosti a ďalšie. Cieľom je predpovedať, či bude na ďalší deň pršať.

# Importy a načítanie dát

In [1]:
# chyby
import os
os.environ['OMP_NUM_THREADS'] = '1'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import folium
import os; os.makedirs('outputs', exist_ok=True) # folder pre grafy
import warnings
warnings.filterwarnings('ignore')

# pre clustering
from sklearn.cluster import KMeans

# missing data
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Nastavenie štýlu grafov
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.autolayout'] = True

# Zobrazovať všetky stĺpce v pandas
pd.set_option('display.max_columns', 30)

C:\ProgramData\miniconda3\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.5)
  from scipy.stats import gaussian_kde


In [3]:
# === Pomocné funkcie ===

def plot_histogram(ax, data, title, xlabel='', ylabel='Počet dní',
                   color='#3498db', bins=80, stats_lines=None):
    """Vykreslí histogram s voliteľnými štatistickými čiarami (priemer, medián...)."""
    ax.hist(data, bins=bins, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    if stats_lines is None:
        stats_lines = {'Priemer': ('red', '--', data.mean()),
                       'Medián': ('orange', '--', data.median())}

    for label, (clr, ls, val) in stats_lines.items():
        ax.axvline(val, color=clr, linestyle=ls, label=f'{label}: {val:.2f}')

    ax.legend()

In [4]:
# Globalne nastavenie
# ci ukladat grafy
save_fig = True

In [ ]:
# Načítanie dát z Excel súboru
df_raw = pd.read_excel('https://github.com/danakozakova/MachineLearningProjekt/raw/main/data/australia_weather.xlsx', header=None, skiprows=10)

# len relevantné stĺpce (index 2 až 25)
df = df_raw.iloc[:, 2:26].copy()

# správne názvy stĺpcov (podľa description.txt a hlavičky z Excelu)
df.columns = [
    'Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
    'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm',
    'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
    'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm',
    'Temp9am', 'Temp3pm', 'RainToday', 'RISK_MM', 'RainTomorrow' 
]

In [ ]:
# odstrániť úplne prázdne riadky
df = df.dropna(how='all')

# Odstrániť riadky s pivotovými sumármi
df = df[~df['Date'].astype(str).str.contains('Součet|Celkem|prázdné', na=False)]
df = df[~df['Location'].astype(str).str.contains('Součet|Celkem|prázdné', na=False)]

# Konverzia dátumu
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')

# Konverzia YES/NO
df['RainToday'] = df['RainToday'].replace({'Yes': 1, 'No': 0})
df['RainTomorrow'] = df['RainTomorrow'].replace({'Yes': 1, 'No': 0})

In [ ]:
# Informácie o stĺpcoch a ich dátových typoch
df.info()

In [ ]:
df.shape

# Základný prehľad dát


## Rozmery a typy dát

In [ ]:
# Veľkosť datasetu
print(f'Veľkosť datasetu: {df.shape}')
print(f'Časové rozpätie: {df["Date"].min().date()} až {df["Date"].max().date()}')
print(f'Počet unikátnych lokalít: {df["Location"].nunique()}')

In [ ]:
# Informácie o stĺpcoch a ich dátových typoch
df.info()

In [ ]:
# Základné štatistiky pre číselné stĺpce
numeric_cols = df.select_dtypes("number").columns.tolist()
df[numeric_cols].describe().round(2)

In [ ]:
# Základné štatistiky pre kategorialne stĺpce
categorical_cols = df.select_dtypes("object").columns.tolist()
df[categorical_cols].describe()

- Dataset obsahuje **~156 tisíc** denných záznamov z **49 lokalít** v Austrálii za obdobie november 2007 až jún 2017.
- Dataset má **24 stĺpcov**: 17 číselných, 4 kategorické (smery vetra), 2 binárne (RainToday, RainTomorrow) a 1 dátum.
- Najzastúpenejšia lokalita je **Canberra** (~3 800 záznamov), najčastejší smer vetra je **W** (západ).

## Chýbajúce hodnoty

In [ ]:
# Počet a percentuálny podiel chýbajúcich hodnôt
missing = pd.DataFrame({
    'Chýbajúce hodnoty': df.isnull().sum(),
    'Podiel (%)': (df.isnull().mean() * 100).round(2)
})
missing = missing.sort_values('Podiel (%)', ascending=False)

In [ ]:
missing_nonzero = (missing[missing['Podiel (%)'] > 0]
                   .sort_values('Podiel (%)', ascending=True))

# farba podľa závažnosti
def severity_color(pct):
    if pct > 30: return 'red'  
    return 'blue'           

colors = missing_nonzero['Podiel (%)'].map(severity_color)


In [ ]:
# Graf
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(missing_nonzero.index, missing_nonzero['Podiel (%)'], color=colors)
ax.set_xlabel('Podiel chýbajúcich hodnôt (%)')
ax.set_title('Chýbajúce hodnoty v stĺpcoch')

for bar, pct in zip(bars, missing_nonzero['Podiel (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/graf_missing_values.png', bbox_inches='tight')
plt.show()

- **Kriticky chýbajúce stĺpce** (>30 %): `Sunshine` (47.7 %), `Evaporation` (42.8 %), `Cloud3pm` (40.1 %), `Cloud9am` (37.7 %). Tieto stĺpce pravdepodobne nie sú merané vo všetkých staniciach.
- **Mierne chýbajúce** (5–10 %): `Pressure9am/3pm` (~ 9.8 %), `WindDir9am` (7 %), `WindGustDir/Speed` (~ 6.5 %).
- **Minimálne chýbajúce** (<3 %): väčšina ostatných stĺpcov.
- **Cieľové premenné** `RainTomorrow` a `RISK_MM` nemajú žiadne chýbajúce hodnoty.


## Distribúcie - boxploty

### Distribúcia cieľovej premennej

Aký je pomer dní s dažďom vs. bez dažďa? Vyvážený / nevyvážený dataset?

In [ ]:
# Distribúcia binomickej cieľovej premennej
target_summary = (df['RainTomorrow']
                  .value_counts()
                  .to_frame('Počet')
                  .assign(**{'Podiel (%)': lambda x: (x['Počet'] / x['Počet'].sum() * 100).round(1)}))

target_summary.index.name = 'RainTomorrow'
display(target_summary)

- Dataset je **nevyvážený**: ~77.6 % dní je bez dažďa, ~22.4 % dní s dažďom. Pomer je približne **3.5:1**.
- Toto nie je extrémna nevyváženosť, ale samotná *accuracy* nebude dostatočnou metrikou.

In [ ]:
# Distribúcia RISK_MM (spojitý target)
risk_nonzero = df.loc[df['RISK_MM'] > 0, 'RISK_MM']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_histogram(axes[0], df['RISK_MM'],
               title='Distribúcia RISK_MM (zrážky zajtra v mm)',
               xlabel='RISK_MM (mm)')

plot_histogram(axes[1], risk_nonzero,
               title=f'RISK_MM > 0 (len daždivé dni, n={len(risk_nonzero)})',
               xlabel='RISK_MM (mm)', color='#e74c3c',
               stats_lines={'Priemer': ('darkred', '--', risk_nonzero.mean())})

plt.savefig('outputs/graf_RISK_MM.png', bbox_inches='tight')
plt.show()

In [ ]:
# Štatistiky pre RISK_MM
risk = df['RISK_MM']
pd.DataFrame({
    'Priemer (mm)':   [f'{risk.mean():.2f}'],
    'Medián (mm)':    [f'{risk.median():.2f}'],
    'Max (mm)':       [f'{risk.max():.1f}'],
    'RISK_MM = 0':         [f'{(risk == 0).sum():,} ({(risk == 0).mean()*100:.1f}%)'],
}, index=['RISK_MM'])

**Zistenia — RISK_MM (spojitý target):**

- RISK_MM je silne pravostranné skosená — väčšina dní má **nulové zrážky** (~64 %).
- Priemer (~2.4 mm) je oveľa vyšší než medián (0 mm).
- Ak uvažujeme len daždivé dni (RISK_MM > 1), priemer je výrazne vyšší.
- Pre regresné modely bude pravdepodobne vhodné logaritmovať alebo inak transformovať túto premennú.


In [ ]:
# Distribúcia log(RISK_MM) (spojitý target pre regresiu)
risk_log = np.log(df['RISK_MM'] + 1)
risk_nonzero_log = np.log(df.loc[df['RISK_MM'] >= 1, 'RISK_MM'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_histogram(axes[0], risk_log,
               title='Distribúcia log(RISK_MM)',
               xlabel='log(RISK_MM + 1)')

plot_histogram(axes[1], risk_nonzero_log,
               title=f'log(RISK_MM >= 1) (len daždivé dni, n={len(risk_nonzero_log):,})',
               xlabel='log(RISK_MM)', color='#e74c3c')

plt.savefig('outputs/graf_log_RISK_MM.png', bbox_inches='tight')
plt.show()

### Distribúcie jednotlivých premenných

Preskúmame rozdelenie hodnôt pre každú premennú, aby sme identifikovali outliery, tvar distribúcií a ďalšie anomálie.

In [ ]:
# Histogramy pre všetky číselné stĺpce
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    df[col].dropna().hist(bins=40, ax=ax, color='#3498db', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=9)

# Nezobraziť nepoužité subplot-y
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)
    
plt.tight_layout()
plt.suptitle('Histogramy číselných premenných', fontsize=16, y=1.02)
plt.savefig('outputs/graf_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxploty pre identifikáciu outlierov
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.boxplot(data=df, y=col, ax=ax, color='#3498db', fliersize=2)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylabel('')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.suptitle('Boxploty číselných premenných (identifikácia outlierov)', fontsize=16, y=1.02)
plt.savefig('outputs/graf_boxploty.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribúcia kategorických stĺpcov: smery vetra
wind_dirs = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(wind_dirs):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=axes[i], color='#3498db')
    axes[i].set_title(f'Distribúcia {col}', fontsize=12)
    axes[i].tick_params(axis='x', rotation=45, labelsize=8)
    axes[i].set_xlabel('')

plt.savefig('outputs/graf_wind_hist.png', bbox_inches='tight')
plt.show()

- **Rainfall a RISK_MM** sú silne skosené (pravostranné) — väčšina dní má nulové alebo minimálne zrážky, ale existujú extrémne hodnoty (>300 mm).
- **Evaporation** je tiež pravostranné skosená s outliermi.
- **Teploty** (MinTemp, MaxTemp, Temp9am, Temp3pm) majú približne normálne rozdelenie.
- **Humidity9am** je ľavostranné skosená (vyššie hodnoty), **Humidity3pm** je viac rovnomerná.
- **Pressure** (9am aj 3pm) má podozrivé extrémne hodnoty max ~10 340 hPa — toto je zjavne chyba v dátach (normálny tlak je okolo 1 010–1 030 hPa). Bude treba tieto outliery ošetriť.
- **Cloud** premenné (oktas 0–8) by mali mať hodnoty 0–8, ale vyskytujú sa aj hodnoty 9 (neurčité) — to je malá anomália.
- **Smery vetra:** Najprevládajúcejšie smery sú **W** (západ) a **N** (sever) pre 9:00 ráno, **SE** (juhovýchod) pre 15:00.

## Porovnanie podľa lokality a sezóny

Preskúmame, či existujú výrazné rozdiely medzi lokalitami a či sa prejavujú sezónne vzory.

In [ ]:
# Podiel daždivých dní podľa lokality
rain_by_location = df.groupby('Location')['RainTomorrow'].apply(
    lambda x: (x == 1).mean() * 100
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
rain_by_location.plot(kind='barh', color='#3498db', edgecolor='white', ax=ax)
ax.set_xlabel('Podiel dažďových dní (%)', fontsize=12)
ax.set_title('Podiel daždivých dní podľa lokality', fontsize=14)
ax.axvline(x=rain_by_location.mean(), color='red', linestyle='--', label=f'Priemer ({rain_by_location.mean():.1f}%)')
ax.legend(fontsize=11)
plt.tight_layout()
import os; os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/graf_rain_locality.png', bbox_inches='tight')
plt.show()

In [ ]:
# Sezónne vzory — mesačné priemery vybraných premenných
df['Month'] = df['Date'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

seasonal_vars = ['Rainfall', 'MaxTemp', 'Humidity3pm', 'Sunshine']
colors_s = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, (col, color) in enumerate(zip(seasonal_vars, colors_s)):
    ax = axes[i // 2, i % 2]
    monthly = df.groupby('Month')[col].mean()
    monthly.plot(ax=ax, marker='o', color=color, linewidth=2, markersize=6)
    ax.set_title(f'Priemerné {col} podľa mesiaca', fontsize=12)
    ax.set_xlabel('Mesiac')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'Máj', 'Jún',
                        'Júl', 'Aug', 'Sep', 'Okt', 'Nov', 'Dec'])

plt.suptitle('Sezónne vzory vo vybraných premenných', fontsize=16, y=1.02)
plt.tight_layout()
import os; os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/graf_monthly.png', bbox_inches='tight')
plt.show()

In [ ]:
# Mesačný podiel daždivých dní
rain_by_month = df.groupby('Month')['RainTomorrow'].apply(
    lambda x: (x == 1).mean() * 100
)

fig, ax = plt.subplots(figsize=(10, 5))
rain_by_month.plot(kind='bar', color='#3498db', edgecolor='white', ax=ax)
ax.set_xlabel('Mesiac', fontsize=12)
ax.set_ylabel('Podiel dažďových dní (%)', fontsize=12)
ax.set_title('Podiel dní s dažďom podľa mesiaca', fontsize=14)
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'Máj', 'Jún',
                     'Júl', 'Aug', 'Sep', 'Okt', 'Nov', 'Dec'], rotation=0)
ax.axhline(y=rain_by_month.mean(), color='red', linestyle='--',
           label=f'Priemer ({rain_by_month.mean():.1f}%)')
ax.legend(fontsize=11)
plt.tight_layout()
import os; os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/graf_rain_monthly.png', bbox_inches='tight')
plt.show()

- **Lokality:** Existujú veľké rozdiely medzi lokalitami — niektoré (napr. pobrežné mestá) majú výrazne vyšší podiel dažďových dní než vnútrozemské a púštne oblasti.
- **Sezónne vzory:**
  - **Zrážky** sú najvyššie v letných mesiacoch (január–marec) a najnižšie v zime (jún–august). Pozor: Austrália je na južnej pologuli, takže sezóny sú opačné!
  - **Teploty** logicky kopírujú letnú/zimnú sezónu.
  - **Slnečný svit** je najvyšší v lete a najnižší v zime.
  - **Vlhkosť** je pomerne stabilná, ale mierne stúpa v zimných mesiacoch.
- Stĺpec `Month` môže byť užitočný feature pri trénovaní modelov (sezónnosť zrážok).

# Oprava chybných hodnôt

## Oprava hodnôt tlaku

Identifikovali sme podozrivo vysoké hodnoty v stĺpcoch `Pressure9am` a `Pressure3pm` (max ~10 340 hPa). Normálny atmosférický tlak sa pohybuje okolo 950–1 050 hPa. Hodnoty nad 9 500 sú zjavne chybné — pravdepodobne ide o chybu pri zápise (chýbajúca desatinná čiarka). Opravíme ich vydelením 10.


In [ ]:
# Počet chybných hodnôt pred opravou
print('Pred opravou:')
print(f'  Pressure9am > 9500: {(df["Pressure9am"] > 9500).sum()} hodnôt')
print(f'  Pressure3pm > 9500: {(df["Pressure3pm"] > 9500).sum()} hodnôt')
print(f'  Pressure9am max: {df["Pressure9am"].max()}')
print(f'  Pressure3pm max: {df["Pressure3pm"].max()}')

In [ ]:
# Oprava: hodnoty > 9500 vydelíme 10
df.loc[df['Pressure9am'] > 9500, 'Pressure9am'] = df.loc[df['Pressure9am'] > 9500, 'Pressure9am'] / 10
df.loc[df['Pressure3pm'] > 9500, 'Pressure3pm'] = df.loc[df['Pressure3pm'] > 9500, 'Pressure3pm'] / 10

print('\nPo oprave:')
print(f'  Pressure9am rozsah: {df["Pressure9am"].min():.1f} – {df["Pressure9am"].max():.1f} hPa')
print(f'  Pressure3pm rozsah: {df["Pressure3pm"].min():.1f} – {df["Pressure3pm"].max():.1f} hPa')

## Oprava hodnôt Cloud

* Hodnoty 9 pre Cloud sú nedefinované, tlak je od 0 po 8, 
* **9** interpretujeme ako **NaN**

In [ ]:
df["Cloud9am"] = df["Cloud9am"].replace(9, np.nan)
df["Cloud3pm"] = df["Cloud3pm"].replace(9, np.nan)

## Podozrivé nuly

In [ ]:
(df == 0).sum().sort_values(ascending=False)

Legitímne nuly
* **Rainfall, RISK_MM, RainToday, RainTomorrow** – veľa dní jednoducho neprší
* **WindSpeed9am/3pm, Cloud9am/3pm – bezvetrie** a jasná obloha sú bežné

In [ ]:
(df.drop(columns = ['RainTomorrow', 'Rainfall', 'RainToday', 'RISK_MM', 'Cloud9am', 'Cloud3pm', 'WindSpeed9am', 'WindSpeed3pm']) == 0).sum().sort_values(ascending= False)

In [ ]:
# Početnosť núl
zero_cols = ['Sunshine', 'Evaporation', 'MaxTemp', 'Temp9am', 'Temp3pm']

zero_summary = pd.DataFrame({
    'Počet núl':  (df[zero_cols] == 0).sum(),
    'Podiel (%)': ((df[zero_cols] == 0).mean() * 100).round(2)
})

zero_summary.index.name = 'Stĺpec'
display(zero_summary)


Tie početnosti sú pomerne malé, takže ich nebudeme riešiť

# Ďalšie úpravy a transformácie

## Radarový graf smerov vetra

Smery vetra zodpovedajú svetovým stranám. Radarový (polárny) graf umožňuje prirodzene vizualizovať frekvenciu jednotlivých smerov. Zobrazíme aj rozdiely medzi daždivými dňami a dňami bez dažďa.


In [ ]:
# Poradie svetových strán (v smere hodinových ručičiek od severu)
compass_order = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
                 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']

# Uhly pre každý smer (v radiánoch)
angles = np.linspace(0, 2 * np.pi, len(compass_order), endpoint=False).tolist()
angles += angles[:1]  # uzavrieť kruh

def radar_wind(col, title):
    fig, ax = plt.subplots(figsize=(4, 4), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi / 2)   # otočí N na vrch (o 90° proti smeru hodinových ručičiek)
    ax.set_theta_direction(-1)        # smery pôjdu v smere hodinových ručičiek (ako kompas)

    for label, color, ls in [(0, '#2ecc71', '-'), (1, '#e74c3c', '--')]:
        subset = df[df['RainTomorrow'] == label]
        counts = subset[col].value_counts()
        # Normalizácia na percentá
        values = [(counts.get(d, 0) / len(subset)) * 100 for d in compass_order]
        values += values[:1]
        ax.plot(angles, values, color=color, linewidth=2, linestyle=ls, label=f'RainTomorrow={label}')
        ax.fill(angles, values, color=color, alpha=0.1)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(compass_order, fontsize=9)
    ax.set_title(title, fontsize=13, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    return fig

fig1 = radar_wind('WindGustDir', 'Smer najsilnejšieho poryvu vetra')
plt.savefig('outputs/graf_windGust.png', bbox_inches='tight')
plt.show()

fig2 = radar_wind('WindDir9am', 'Smer vetra o 9:00')
plt.savefig('outputs/graf_wind9am.png', bbox_inches='tight')
plt.show()

fig3 = radar_wind('WindDir3pm', 'Smer vetra o 15:00')
plt.savefig('outputs/graf_wind3pm.png', bbox_inches='tight')
plt.show()


- Radarové grafy ukazujú, že prevládajúce smery vetra sa líšia medzi daždivými a bezdaždivými dňami.
- Dažďové dni majú tendenciu mať viac vetra zo **západných a severozápadných** smerov, zatiaľ čo dni bez dažďa majú častejšie **východné a juhovýchodné** smery.
- Toto zodpovedá meteorologickým vzorcom v Austrálii — vlhký vzduch prichádza zvyčajne od oceánu (západ/severozápad).

## Sezónne vzory podľa dňa v roku

Namiesto mesiacov sa pozrieme na jemnejšie sezónne vzory pomocou poradového čísla dňa v roku (1–366). Pre prehľadnosť použijeme kĺzavý priemer.


In [ ]:
# Výpočet dňa v roku
df['DayOfYear'] = df['Date'].dt.dayofyear

# Kĺzavý priemer pre vybrané premenné
seasonal_vars = ['Rainfall', 'MaxTemp', 'Humidity3pm', 'Sunshine', 'Pressure3pm', 'Evaporation']
window = 15  # 15-dňový kĺzavý priemer

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()
colors_s = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#e74c3c']

for i, (col, color) in enumerate(zip(seasonal_vars, colors_s)):
    daily_mean = df.groupby('DayOfYear')[col].mean()
    rolling = daily_mean.rolling(window=window, center=True, min_periods=1).mean()

    axes[i].plot(daily_mean.index, daily_mean.values, alpha=0.2, color=color, linewidth=0.5)
    axes[i].plot(rolling.index, rolling.values, color=color, linewidth=2.5, label=f'{window}-dňový priemer')
    axes[i].set_title(f'{col} podľa dňa v roku', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Deň v roku')
    axes[i].legend()

plt.tight_layout()
plt.suptitle('Sezónne vzory podľa dňa v roku (kĺzavý priemer)', fontsize=16, y=1.02)
plt.savefig('outputs/graf_seasonability.png', bbox_inches='tight')
plt.show()


In [ ]:
# RISK_MM podľa dňa v roku
fig, ax = plt.subplots(figsize=(14, 5))

risk_daily = df.groupby('DayOfYear')['RISK_MM'].mean()
risk_rolling = risk_daily.rolling(window=window, center=True, min_periods=1).mean()

ax.plot(risk_daily.index, risk_daily.values, alpha=0.15, color='#3498db', linewidth=0.5)
ax.plot(risk_rolling.index, risk_rolling.values, color='#3498db', linewidth=2.5, label=f'{window}-dňový priemer')
ax.set_title('Priemerné RISK_MM podľa dňa v roku', fontsize=14)
ax.set_xlabel('Deň v roku')
ax.set_ylabel('RISK_MM (mm)')
ax.legend()
plt.tight_layout()

plt.savefig('outputs/graf_risk_MM_seasonability.png', bbox_inches='tight')
plt.show()


## Sinus a cosinus pre smer vetra

In [ ]:
def encode_wind_direction(df, col_name):
    """
    Zakóduje kategorický smer vetra na dve číselné premenné (sínus a kosínus).
    """
    # mapovanie smerov vetra na stupne
    wind_mapping = {
        'N': 0.0,   'NNE': 22.5,  'NE': 45.0,   'ENE': 67.5,
        'E': 90.0,  'ESE': 112.5, 'SE': 135.0,  'SSE': 157.5,
        'S': 180.0, 'SSW': 202.5, 'SW': 225.0,  'WSW': 247.5,
        'W': 270.0, 'WNW': 292.5, 'NW': 315.0,  'NNW': 337.5
    }
    
    degrees = df[col_name].map(wind_mapping)

    # prevod stupňov na radiány
    deg_to_rad = np.deg2rad(degrees)
    
    # výpočet sínusu a kosínusu
    wind_sin = np.sin(deg_to_rad)
    wind_cos = np.cos(deg_to_rad)
    
    return degrees, wind_sin, wind_cos

In [ ]:
wind_columns = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

for col in wind_columns:
    df[f'{col}_degree'], df[f'{col}_sin'], df[f'{col}_cos'] = encode_wind_direction(df, col)

## Zmeny vetra, teploty

In [ ]:
# Scatter plot: Humidity3pm vs. Pressure3pm farebne podľa RainTomorrow
fig, ax = plt.subplots(figsize=(7, 4))

# Vzorkovanie pre prehľadnosť (príliš veľa bodov by bolo neprehľadné)
sample = df.dropna(subset=['Humidity3pm', 'Pressure3pm', 'RainTomorrow']).sample(10000, random_state=42)

colors_map = {0: '#2ecc71', 1: '#e74c3c'}
for label, color in colors_map.items():
    subset = sample[sample['RainTomorrow'] == label]
    ax.scatter(subset['Humidity3pm'], subset['Pressure3pm'],
               c=color, label=label, alpha=0.4, s=15, edgecolors='none')

ax.set_xlabel('Humidity3pm (%)', fontsize=12)
ax.set_ylabel('Pressure3pm (hPa)', fontsize=12)
ax.set_title('Humidity3pm vs. Pressure3pm podľa RainTomorrow', fontsize=13)
ax.legend(title='RainTomorrow', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/graf_humidity_pressure.png', bbox_inches='tight')
plt.show()

In [ ]:
df['WindSpeed_change'] = df.WindSpeed3pm - df.WindSpeed9am
df['WindDir_degree_change'] = (df.WindDir3pm_degree - df.WindDir9am_degree + 180) % 360 - 180 # rozsah (-180, 180]
df['Pressure_change'] = df.Pressure3pm - df.Pressure9am
df['Cloud_change'] = df.Cloud3pm - df.Cloud9am
df['Temp_change'] = df.Temp3pm - df.Temp9am
df['Humidity_change'] = df.Humidity3pm - df.Humidity9am
df['Humidity_Pressure9'] = df.Humidity9am * df.Pressure9am
df['Humidity_Pressure3'] = df.Humidity3pm * df.Pressure3pm
df['Humidity_Pressure_change'] = df.Humidity_Pressure3 - df.Humidity_Pressure9

## Logaritmické hodnoty

In [ ]:
df['Rainfall_log'] = np.log(df.Rainfall+0.1)
df['Evaporation_log'] = np.log(df.Evaporation+0.1)
df['WindGustSpeed_log'] = np.log(df.WindGustSpeed+0.1)

# Kontrola rozdelení a vzťahov po oprave dát

## Distribúcia po oprave


In [ ]:
# Kompletné histogramy všetkých číselných premenných po oprave
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()
plot_cols = [c for c in numeric_cols if c != 'RISK_MM']

for i, col in enumerate(plot_cols):
    ax = axes[i]
    df[col].dropna().hist(bins=40, ax=ax, color='#2ecc71', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.tick_params(labelsize=9)

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.suptitle('Histogramy číselných premenných (po oprave tlaku)', fontsize=16, y=1.02)
plt.savefig('outputs/graf_histogramy_oprava.png', bbox_inches='tight')
plt.show()


## Korelácie po oprave dát

In [ ]:
# Korelačná matica po oprave
corr_matrix_fixed = df[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix_fixed, dtype=bool))

sns.heatmap(corr_matrix_fixed, mask=mask, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            annot_kws={'size': 8})
ax.set_title('Korelačná matica (po oprave dát)', fontsize=16, pad=20)

plt.savefig('outputs/graf_correl_matrix_oprava.png', bbox_inches='tight')
plt.show()


- **Silne korelované páry** (redundancia):
  - `MinTemp` ↔ `Temp9am` (~0.9) — logické, ranná teplota závisí od nočného minima.
  - `MaxTemp` ↔ `Temp3pm` (~0.98) — popoludňajšia teplota je takmer totožná s denným maximom.
  - `Pressure9am` ↔ `Pressure3pm` (~0.96) — tlak sa počas dňa príliš nemení.
  - `RISK_MM` ↔ `Rainfall` — silná pozitívna korelácia.
  - Pri feature selection by stačilo ponechať len jednu premennú z každého silne korelovaného páru. Alebo dať neskorší údaj (3pm) a zmenu medzi 9am a 3pm

In [ ]:
# Boxploty kľúčových premenných vs. RainTomorrow 
key_features = ['Humidity3pm', 'Humidity_change',
                'Pressure3pm', 'Pressure_change',
                'Temp3pm', 'Temp_change', 'Sunshine', 
                'Cloud3pm', 'Cloud_change',
                'Rainfall', 'WindGustSpeed', 'WindSpeed_change', 'WindDir_degree_change',
                'Evaporation', ]

fig, axes = plt.subplots(5, 3, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.boxplot(data=df, x='RainTomorrow', y=col, ax=axes[i],
                palette={0: '#2ecc71', 1: '#e74c3c'}, 
                hue = 'RainTomorrow',
                legend = False)
    axes[i].set_title(f'{col} vs. RainTomorrow', fontsize=11)
    axes[i].set_xlabel('')

plt.suptitle('Kľúčové premenné vs. RainTomorrow', fontsize=16, y=1.02)
plt.tight_layout()

plt.savefig('outputs/graf_boxploty_oprava.png', bbox_inches='tight')
plt.show()

- **Najlepšie prediktory `RainTomorrow`** (podľa boxplotov):
  - **Humidity3pm** — jasný rozdiel medzi triedami. Daždivé dni majú výrazne vyššiu vlhkosť (~70–80 % vs. ~40–50 %).
  - **Sunshine** — menej hodín slnka v dňoch pred dažďom.
  - **Cloud3pm** — vyššia oblačnosť pred dažďom.
  - **Pressure3pm** — nižší tlak pred dažďom.
  - **WindGustSpeed** — silnejšie poryvy vetra pred dažďom.

# Imputácia chýbajúcich hodnôt

## Chýbajúce hodnoty a ich vzťah k cieľovým premenným

Analyzujeme, či chýbanie hodnôt v jednotlivých stĺpcoch súvisí s cieľovými premennými, či inými premennými. Ak áno, chýbanie nie je náhodné (MAR/MNAR) a treba to zohľadniť pri imputácii.


In [ ]:
# Pre každý stĺpec s chýbajúcimi hodnotami porovnáme target distribúciu
cols_with_missing = [c for c in df.columns if df[c].isnull().sum() > 0 and c not in ['Date', 'Month', 'DayOfYear']]

# RainTomorrow: porovnanie podielu Yes pre riadky s/bez chýbajúcich hodnôt
results = []
for col in cols_with_missing:
    missing_mask = df[col].isnull()
    miss_pct = missing_mask.mean() * 100

    rain_when_missing = (df.loc[missing_mask, 'RainTomorrow'] == 1).mean() * 100 if missing_mask.sum() > 0 else np.nan
    rain_when_present = (df.loc[~missing_mask, 'RainTomorrow'] == 1).mean() * 100

    risk_when_missing = df.loc[missing_mask, 'RISK_MM'].mean() if missing_mask.sum() > 0 else np.nan
    risk_when_present = df.loc[~missing_mask, 'RISK_MM'].mean()

    results.append({
        'Stĺpec': col,
        'Chýba (%)': round(miss_pct, 1),
        'RainTom=Yes keď chýba (%)': round(rain_when_missing, 1) if not np.isnan(rain_when_missing) else '-',
        'RainTom=Yes keď nechýba (%)': round(rain_when_present, 1),
        'RISK_MM keď chýba (mm)': round(risk_when_missing, 2) if not np.isnan(risk_when_missing) else '-',
        'RISK_MM keď nechýba (mm)': round(risk_when_present, 2)
    })

results_df = pd.DataFrame(results).sort_values('Stĺpec', ascending=False)
display(results_df)

In [ ]:
sig_df = results_df[(results_df['Chýba (%)'] > 0.5) & 
                    (~results_df['Stĺpec'].str.contains('log'))].copy()

In [ ]:
sig_df = results_df[(results_df['Chýba (%)'] > 0.5) & 
                    (~results_df['Stĺpec'].str.contains('log'))].copy()

plt.figure(figsize=(10, 8))
labels = sig_df['Stĺpec'].values
y_pos = np.arange(len(labels))
width = 0.35

plt.barh(y_pos - width/2, sig_df['RainTom=Yes keď nechýba (%)'], width, label='Existing', color='#2ecc71')
plt.barh(y_pos + width/2, sig_df['RainTom=Yes keď chýba (%)'], width, label='Missing', color='#e74c3c')

for i, pct in enumerate(sig_df['Chýba (%)'].values):
    plt.text(max(sig_df['RainTom=Yes keď chýba (%)'].iloc[i], sig_df['RainTom=Yes keď nechýba (%)'].iloc[i]), 
             y_pos[i], f'({pct:.0f}% chýba)', va='center', fontsize=8, color='gray')

plt.yticks(y_pos, labels)
plt.xlabel('RainTomorrow - priemer')
plt.title('RainTomorrow vs. chýbajúce hodnoty', fontsize=13)
plt.legend()
plt.savefig('outputs/graf_RainTomorrow_missing.png', bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
labels = sig_df['Stĺpec'].values
y_pos = np.arange(len(labels))
width = 0.35

plt.barh(y_pos - width/2, sig_df['RISK_MM keď nechýba (mm)'], width, label='Existing', color='#2ecc71')
plt.barh(y_pos + width/2, sig_df['RISK_MM keď chýba (mm)'], width, label='Missing', color='#e74c3c')

for i, pct in enumerate(sig_df['Chýba (%)'].values):
    plt.text(max(sig_df['RISK_MM keď chýba (mm)'].iloc[i], sig_df['RISK_MM keď nechýba (mm)'].iloc[i]),
             y_pos[i], f'({pct:.0f}% chýba)', va='center', fontsize=8, color='gray')

plt.yticks(y_pos, labels)
plt.xlabel('Priemerné RISK_MM (mm)')
plt.title('RISK_MM vs. chýbajúce hodnoty', fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig('outputs/graf_risk_mm_missing.png', bbox_inches='tight')
plt.show()

## Chýbajúce hodnoty a ich vzťah ku staniciam

In [ ]:
# Pôvodné premenné (bez odvodených)
original_cols = ['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine',
                 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm',
                 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
                 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm',
                 'Temp9am', 'Temp3pm', 'RainToday']

# Percentá chýbajúcich hodnôt: riadky = stanice, stĺpce = premenné
missing_by_station = df.groupby('Location')[original_cols].apply(
    lambda x: x.isnull().mean() * 100
).round(1)

# Heatmapa
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(missing_by_station, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Chýba (%)'},
            vmin=0, vmax=100, ax=ax)
ax.set_title('Podiel chýbajúcich hodnôt (%) podľa staníc', fontsize=16, fontweight='bold')
ax.set_xlabel('Premenná')
ax.set_ylabel('Stanica')
plt.tight_layout()
plt.savefig('outputs/missing_by_station.png', dpi=150, bbox_inches='tight')
plt.show()


### Pripojit súradnice

In [ ]:
# Načítanie súboru so stanicami
stations_df = pd.read_csv('https://github.com/danakozakova/MachineLearningProjekt/raw/main/data/bom_stations.csv')


In [ ]:
# Pomocná funkcia na úpravu názvov (z 'AliceSprings' urobí 'ALICE SPRINGS')
def normalize_location(loc):
     # Pridá medzeru iba medzi malým a veľkým písmenom (napr. 'e' a 'R' v PearceRAAF)
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', str(loc)).upper()

# Vytvoríme si dočasný stĺpec v našom hlavnom datasete
df['Loc_Match'] = df['Location'].apply(normalize_location)

In [ ]:
# Funkcia na nájdenie súradníc
def find_coordinates(loc_name, stations):
    # Vyhľadáme všetky stanice, ktoré obsahujú náš upravený názov
    matches = stations[stations['Site_name'].str.contains(loc_name, na=False)]
    if not matches.empty:
        # Vrátime priemer súradníc (Lat, Lon) všetkých nájdených staníc pre túto lokalitu
        return matches['Lat'].mean(), matches['Lon'].mean()
    return pd.NA, pd.NA

In [ ]:
# Nájdenie súradníc pre jedinečné lokality 
unique_locations = df['Loc_Match'].unique()
coords_dict = {}
for loc in unique_locations:
    coords_dict[loc] = find_coordinates(loc, stations_df)
# Vytvoríme z nájdených súradníc tabuľku
coords_df = pd.DataFrame.from_dict(coords_dict, orient='index', columns=['Lat', 'Lon']).reset_index()
coords_df.rename(columns={'index': 'Loc_Match'}, inplace=True)

In [ ]:
# Pripojenie (Merge) do hlavného datasetu
df = pd.merge(df, coords_df, on='Loc_Match', how='left')
# zmazanie pomocného stĺpca
df.drop(columns=['Loc_Match'], inplace=True)

In [ ]:
# Kontrola výsledku
print("Počet chýbajúcich súradníc po spojení:")
print(df[['Lat', 'Lon']].isnull().sum())

In [ ]:
# Vyfiltrujeme riadky, kde Lat chýba (je NaN) a vypíšeme jedinečné názvy lokalít
chybajuce_lokality = df[df['Lat'].isnull()]['Location'].unique()

print(f"Lokalít bez súradníc je {len(chybajuce_lokality)}.")
print(f"Tu je ich zoznam:{chybajuce_lokality}")

### Stanice na mape Australie

In [ ]:
# jedinečné lokality a ich súradnice (odstránime tie, ktorým súradnice chýbajú)
map_data = df[['Location', 'Lat', 'Lon']].dropna().drop_duplicates()

In [ ]:
# mapa, vycentrovanúá približne na stred Austrálie s počiatočným zoomom
m = folium.Map(location=[-25.2744, 133.7751], zoom_start=4, tiles='OpenStreetMap')

In [ ]:
# body (markery) na mape
for idx, row in map_data.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Lon']],
        popup=row['Location'],   # text po kliknutí
        tooltip=row['Location'], # text pri prejdení myšou
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(m)

In [ ]:
# uložiť mapu do priečinka outputs
m.save('outputs/mapa_stanic.html')

m

## Clustering podľa súradníc

In [ ]:
# K-Means clustering
n_clusters = 9

map_data = df[['Location', 'Lat', 'Lon']].dropna().drop_duplicates(subset=['Location'])

scaler = StandardScaler()
coords_scaled = scaler.fit_transform(map_data[['Lat', 'Lon']])

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
map_data['Region'] = kmeans.fit_predict(coords_scaled)

In [ ]:
# Farby pre jednotlivé regióny
region_colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'gray', 'darkblue', 'darkgreen']

m = folium.Map(location=[-25.2744, 133.7751], zoom_start=4, tiles='OpenStreetMap')

for _, row in map_data.iterrows():
    region = int(row['Region'])
    folium.CircleMarker(
        location=[row['Lat'], row['Lon']],
        radius=8,
        color=region_colors[region],
        fill=True,
        fill_color=region_colors[region],
        fill_opacity=0.7,
        popup=f"{row['Location']} (Región {region})",
        tooltip=f"{row['Location']} — Región {region}"
    ).add_to(m)

# Legenda
legend_html = '<div style="position:fixed; bottom:30px; left:30px; background:white; padding:10px; border:2px solid grey; border-radius:5px; z-index:9999; font-size:13px;">'
legend_html += '<b>Regióny</b><br>'
for i in range(map_data['Region'].nunique()):
    count = (map_data['Region'] == i).sum()
    legend_html += f'<i style="background:{region_colors[i]}; width:12px; height:12px; display:inline-block; border-radius:50%; margin-right:5px;"></i> Región {i} ({count} staníc)<br>'
legend_html += '</div>'
m.get_root().html.add_child(folium.Element(legend_html))

# Uloženie
m.save('outputs/mapa_regionov.html')
m


In [ ]:
# aktualizácia regiónu v hlavnom datasete 
if 'Region' in df.columns:
    df.drop(columns=['Region'], inplace=True)
df = df.merge(map_data[['Location', 'Region']], on='Location', how='left')

## Imputacia

In [ ]:
# ──────────────────────────────────────────────
# Vlastný transformer: imputácia podľa regiónu
# ──────────────────────────────────────────────
class RegionalImputer(BaseEstimator, TransformerMixin):
    """
    Imputuje chýbajúce hodnoty mediánom (číselné) alebo modom (kategorické)
    podľa stĺpca 'Region'. Ak celý región má NaN, použije globálnu hodnotu.
    """
    def __init__(self, region_col='Region'):
        self.region_col = region_col
    
    def fit(self, X, y=None):
        # Pre každý región si uložíme medián (num) a modus (cat)
        self.fill_values_ = {}
        
        for region in X[self.region_col].unique():
            mask = X[self.region_col] == region
            region_data = X.loc[mask]
            self.fill_values_[region] = {}
            
            for col in X.columns:
                if col == self.region_col:
                    continue
                if X[col].dtype in ['float64', 'int64', 'float32']:
                    self.fill_values_[region][col] = region_data[col].median()
                else:
                    mode = region_data[col].mode()
                    self.fill_values_[region][col] = mode[0] if len(mode) > 0 else np.nan
        
        # Globálne záložné hodnoty (keby celý región mal NaN)
        self.global_fill_ = {}
        for col in X.columns:
            if col == self.region_col:
                continue
            if X[col].dtype in ['float64', 'int64', 'float32']:
                self.global_fill_[col] = X[col].median()
            else:
                mode = X[col].mode()
                self.global_fill_[col] = mode[0] if len(mode) > 0 else np.nan
        
        return self
    
    def transform(self, X, y=None):
        X = X.copy()
        
        for region in X[self.region_col].unique():
            mask = X[self.region_col] == region
            for col in X.columns:
                if col == self.region_col:
                    continue
                fill_val = self.fill_values_.get(region, {}).get(col, np.nan)
                # Ak regionálna hodnota je NaN, použijeme globálnu
                if pd.isna(fill_val):
                    fill_val = self.global_fill_.get(col, np.nan)
                X.loc[mask, col] = X.loc[mask, col].fillna(fill_val)
        
        return X

In [ ]:
# Stĺpce, ktoré NECHCEME imputovať
skip_cols = ['Location', 'Date', 'Region', 'RainTomorrow', 'RISK_MM']

cols_to_impute = [c for c in df.columns if c not in skip_cols]
cols_with_missing = [c for c in cols_to_impute if df[c].isnull().any()]

# Indikátory chýbania (0/1) — aby model vedel, kde boli pôvodne NaN
for col in cols_with_missing:
    df[f'{col}_imputed'] = df[col].isnull().astype(int)

# Prehľad pred imputáciou
pd.set_option('future.no_silent_downcasting', True)
before = df[cols_with_missing].isnull().sum().rename('Chýba (pred)')

# Imputácia
imputer = RegionalImputer(region_col='Region')
df[cols_to_impute + ['Region']] = imputer.fit_transform(df[cols_to_impute + ['Region']])

# Prehľad po imputácii
after = df[cols_with_missing].isnull().sum().rename('Chýba (po)')

# Súhrnná tabuľka
summary = pd.DataFrame({
    'Chýbalo': before,
    'Podiel (%)': (before / len(df) * 100).round(1),
    'Zostáva': after
}).query('Chýbalo > 0').sort_values('Chýbalo', ascending=False)

display(summary)
print(f'Imputovaných stĺpcov: {len(cols_with_missing)}, '
      f'vytvorených indikátorov: {len(cols_with_missing)}')

In [ ]:
df

# Finálne pohľady

In [ ]:
# Korelácie s targetmi
numeric_cols = df.select_dtypes(include=[np.number]).columns

corr_risk = df[numeric_cols].corr()['RISK_MM'].drop(['RISK_MM', 'RainTomorrow'])
corr_rain = df[numeric_cols].corr()['RainTomorrow'].drop(['RISK_MM', 'RainTomorrow'])

# Zoradené podľa absolútnej hodnoty (najsilnejšia korelácia hore)
corr_df = pd.DataFrame({
    'RISK_MM': corr_risk,
    'RainTomorrow': corr_rain,
    '|RISK_MM|': corr_risk.abs(),
    '|RainTomorrow|': corr_rain.abs()
}).sort_values('|RainTomorrow|', ascending=False)

# Zobrazenie
display(corr_df.style.background_gradient(cmap='RdBu_r', subset=['RISK_MM', 'RainTomorrow'], vmin=-1, vmax=1)
                     .format('{:.3f}'))


## Uloženie datasetu

In [ ]:
df.columns

In [ ]:
column_order = [
    # === Identifikácia ===
    'Date', 'Location', 'Lat', 'Lon', 'Region',
    'Month', 'DayOfYear',

    # === Teplota ===
    'MinTemp', 'MaxTemp', 'Temp9am', 'Temp3pm', 'Temp_change',
    'MinTemp_imputed', 'MaxTemp_imputed', 'Temp9am_imputed', 'Temp3pm_imputed', 'Temp_change_imputed',

    # === Zrážky ===
    'Rainfall', 'Rainfall_log', 'RainToday', 'RISK_MM', 'RainTomorrow',
    'Rainfall_imputed', 'Rainfall_log_imputed', 'RainToday_imputed',

    # === Evaporácia & Slnko ===
    'Evaporation', 'Evaporation_log', 'Sunshine',
    'Evaporation_imputed', 'Evaporation_log_imputed', 'Sunshine_imputed',

    # === Rýchlosť vetra ===
    'WindGustSpeed', 'WindGustSpeed_log', 'WindSpeed9am', 'WindSpeed3pm', 'WindSpeed_change',
    'WindGustSpeed_imputed', 'WindGustSpeed_log_imputed', 'WindSpeed9am_imputed', 'WindSpeed3pm_imputed', 'WindSpeed_change_imputed',

    # === Smer vetra ===
    'WindGustDir', 'WindGustDir_degree', 'WindGustDir_sin', 'WindGustDir_cos',
    'WindDir9am', 'WindDir9am_degree', 'WindDir9am_sin', 'WindDir9am_cos',
    'WindDir3pm', 'WindDir3pm_degree', 'WindDir3pm_sin', 'WindDir3pm_cos',
    'WindDir_degree_change',
    'WindGustDir_imputed', 'WindGustDir_degree_imputed', 'WindGustDir_sin_imputed', 'WindGustDir_cos_imputed',
    'WindDir9am_imputed', 'WindDir9am_degree_imputed', 'WindDir9am_sin_imputed', 'WindDir9am_cos_imputed',
    'WindDir3pm_imputed', 'WindDir3pm_degree_imputed', 'WindDir3pm_sin_imputed', 'WindDir3pm_cos_imputed',
    'WindDir_degree_change_imputed',

    # === Vlhkosť ===
    'Humidity9am', 'Humidity3pm', 'Humidity_change',
    'Humidity9am_imputed', 'Humidity3pm_imputed', 'Humidity_change_imputed',

    # === Tlak ===
    'Pressure9am', 'Pressure3pm', 'Pressure_change',
    'Pressure9am_imputed', 'Pressure3pm_imputed', 'Pressure_change_imputed',

    # === Oblačnosť ===
    'Cloud9am', 'Cloud3pm', 'Cloud_change',
    'Cloud9am_imputed', 'Cloud3pm_imputed', 'Cloud_change_imputed',

    # === Interakcie (vlhkosť × tlak) ===
    'Humidity_Pressure9', 'Humidity_Pressure3', 'Humidity_Pressure_change',
    'Humidity_Pressure9_imputed', 'Humidity_Pressure3_imputed', 'Humidity_Pressure_change_imputed',
]

# Aplikovanie (overí, či sú všetky stĺpce prítomné)
missing = [c for c in column_order if c not in df.columns]
extra = [c for c in df.columns if c not in column_order]

if missing:
    print(f"Chýbajú v df: {missing}")
if extra:
    print(f"Nie sú v zozname: {extra}")

df = df[column_order]
print(f" Stĺpce preusporiadané: {df.shape[1]} stĺpcov")


In [ ]:
# Uloženie upraveného datasetu
df.to_csv('data/australia_weather_transformed.csv', index=False)

---

# Časť 2: Tréning modelov

> *Nasledujúce bunky pochádzajú z notebooku `projekt_modely2.ipynb`*

# Tréning modelov — Australia Weather

**Klasifikácia** (RainTomorrow): Decision Tree, Logistic Regression, Random Forest  
**Regresia** (RISK_MM): Gradient Boosting  
**Feature sety**: selected, no_critical, all

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['OMP_NUM_THREADS'] = '1'

from sklearn.model_selection import StratifiedKFold, KFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, mean_absolute_error,
                             mean_squared_error, r2_score)
from scipy.stats import randint, uniform

In [ ]:
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
os.makedirs('outputs', exist_ok=True)

In [ ]:
df = pd.read_csv('data/australia_weather_transformed.csv')
print(f"Dataset: {df.shape}")

In [ ]:
df.columns

In [ ]:
df_orig = pd.read_csv('data/australia_weather.csv')
print(f"Dataset: {df_orig.shape}")

In [ ]:
df_orig.columns

## Konfigurácia
### Feature sety

In [ ]:
# === Feature sety ===

# Set 1: Vybrané (3pm + zmeny)
features_selected = [
    'Region',
    'MinTemp', 'MaxTemp', 'Temp3pm', 'Temp_change',
    'Rainfall_log', 'RainToday',          # <- log namiesto Rainfall
    'Evaporation_log', 'Sunshine',        # <- log namiesto Evaporation
    'WindGustSpeed_log', 'WindSpeed3pm', 'WindSpeed_change',  # <- log
    'WindDir_degree_change',
    'Humidity3pm', 'Humidity_change',
    'Pressure3pm', 'Pressure_change',
    'Cloud3pm', 'Cloud_change',
    'Month', 'DayOfYear',
]

# Set 2: Bez kritických (Evaporation, Sunshine, Cloud)
features_no_critical = [f for f in features_selected
                        if not any(x in f for x in ['Evaporation', 'Sunshine', 'Cloud'])]

# Set 3: Všetky
exclude = ['RainTomorrow', 'RISK_MM', 'Location', 'Lat', 'Lon']
features_all = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

feature_sets = {
    'selected': features_selected,
    'no_critical': features_no_critical,
    'all': features_all,
}
# Pridať imputed flagy ku každému setu
for name in ['selected', 'no_critical', 'all']:
    base = eval(f'features_{name}')
    flags = [c + '_imputed' for c in base if c + '_imputed' in df.columns]
    exec(f'features_{name} = base + flags')

for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")

print(f'orig: {df_orig.shape[1]} features')


In [ ]:
cat_cols = make_column_selector(dtype_include=object)
num_cols = make_column_selector(dtype_exclude=object)

## Klasifikačné modely

### Targety

In [ ]:
# === Targety ===
y_cls = df['RainTomorrow']

# === CV stratégia ===
cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Pipeline

In [ ]:
# Decision tree
tree_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown = 'ignore'), cat_cols),
        remainder = 'passthrough'
    ),
    DecisionTreeClassifier(random_state = 42)
)

In [ ]:
# Logistic Regression
logistic_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore'), cat_cols),
        (StandardScaler(), num_cols)
    ),
    LogisticRegression(max_iter=1000, random_state=42)
)

In [ ]:
# Random Forest
forestClass_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown = 'ignore'), cat_cols),
        remainder = 'passthrough'
    ),
    RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
)

In [ ]:
## Klasifikácia (RainTomorrow)
clf_models = {
    'DecisionTree': tree_pipeline,
    'LogisticRegression': logistic_pipeline,
    'RandomForest': forestClass_pipeline
}


### Evaluácia modelov

In [ ]:
def eval_classifier(model, X, y, cv):
    """Cross-validated klasifikačné metriky."""
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return {m: scores[f'test_{m}'].mean() for m in scoring}

### Natrénované modely (základné)

In [ ]:
results = []

# --- Klasifikácia ---
for model_name, model in clf_models.items():
    for fs_name, fs_cols in feature_sets.items():
        X = df[fs_cols]
        scores = eval_classifier(model, X, y_cls, cv_cls)
        results.append({'model': model_name, 'features': fs_name, 'target': 'RainTomorrow', **scores})
        print(f"{model_name:20s} | {fs_name:12s} | Acc={scores['accuracy']:.4f}  Prec={scores['precision']:.4f}  Rec={scores['recall']:.4f}  F1={scores['f1']:.4f}  AUC={scores['roc_auc']:.4f}")


### Natrénované modely (základné) na pôvodných dátach

In [ ]:
cat_selector = make_column_selector(dtype_include=object)
num_selector = make_column_selector(dtype_exclude=object)

# Preprocessing pre tree modely (nepotrebujú scaling)
orig_tree_prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_selector),
    (SimpleImputer(strategy='median'), num_selector)
)

# Preprocessing pre lineárne modely (scaling)
orig_linear_prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore')), cat_selector),
    (make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num_selector)
)

# Pipelines
orig_tree_pipeline = make_pipeline(orig_tree_prep, DecisionTreeClassifier(random_state=42))
orig_logistic_pipeline = make_pipeline(orig_linear_prep, LogisticRegression(max_iter=1000, random_state=42))
orig_forest_pipeline = make_pipeline(orig_tree_prep, RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))

In [ ]:
# Data z originalneho datasetu
X = df_orig.drop(columns = ['RISK_MM', 'RainTomorrow'])
y = df_orig.RainTomorrow

In [ ]:
scores = eval_classifier(orig_tree_pipeline, X, y, cv_cls)
results.append({'model': 'DecisionTree', 'features': 'original', 'target': 'RainTomorrow', **scores})
print(f"DecisionTree | original | Acc={scores['accuracy']:.4f}  Prec={scores['precision']:.4f}  Rec={scores['recall']:.4f}  F1={scores['f1']:.4f}  AUC={scores['roc_auc']:.4f}")


In [ ]:
scores = eval_classifier(orig_logistic_pipeline, X, y, cv_cls)
results.append({'model': 'LogisticRegression', 'features': 'original', 'target': 'RainTomorrow', **scores})
print(f"LogisticRegression | original | Acc={scores['accuracy']:.4f}  Prec={scores['precision']:.4f}  Rec={scores['recall']:.4f}  F1={scores['f1']:.4f}  AUC={scores['roc_auc']:.4f}")


## Regresné modely

### TargetY

In [ ]:
# === Targety ===
y_reg = np.log(df['RISK_MM']+0.1)

# === CV stratégia ===
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

### Pipeline

In [ ]:
linear_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown='ignore'), cat_cols),
        (StandardScaler(), num_cols)
    ),
    LinearRegression()
)

In [ ]:
# Random Forest Regressor
forestReg_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown = 'ignore'), cat_cols),
        remainder = 'passthrough'
    ),
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
)

In [ ]:
# GradientBoosting
gradient_pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown = 'ignore'), cat_cols),
        remainder = 'passthrough'
    ),
    HistGradientBoostingRegressor(max_iter=200, random_state=42))

In [ ]:
# Regresia (RISK_MM) — HistGradientBoosting zvláda NaN natívne,
reg_models = {
    'LinearRegression': linear_pipeline,
    'RandomForestRegressor': forestReg_pipeline,
    'GradientBoosting': gradient_pipeline,
}

### Evaluácia modelov

In [ ]:
def eval_regressor(model, X, y_reg, y_cls, cv):
    """Cross-validated regresné metriky + odvodená klasifikácia (>=1mm → dážď)."""
    y_pred = cross_val_predict(model, X, y_reg, cv=cv, n_jobs=1)

    # Regresné metriky
    result = {
        'MAE': mean_absolute_error(y_reg, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_reg, y_pred)),
        'R2': r2_score(y_reg, y_pred),
    }

    # Odvodená klasifikácia: predikcia >= 1mm → RainTomorrow = 1
    y_pred_cls = (y_pred >= 1).astype(int)
    result['derived_f1'] = f1_score(y_cls, y_pred_cls)
    result['derived_accuracy'] = accuracy_score(y_cls, y_pred_cls)

    return result

### Natrénované modely (základné)

In [ ]:
# --- Regresia ---
for model_name, model in reg_models.items():
    for fs_name, fs_cols in feature_sets.items():
        X = df[fs_cols]
        scores = eval_regressor(model, X, y_reg, y_cls, cv_reg)
        results.append({'model': model_name, 'features': fs_name, 'target': 'RISK_MM', **scores})
        print(f"{model_name:20s} | {fs_name:12s} | RMSE={scores['RMSE']:.4f}  R2={scores['R2']:.4f}  derived_F1={scores['derived_f1']:.4f}")


In [ ]:
new_df = pd.DataFrame(results)
file_exists = os.path.exists("outputs/results_training.csv")
new_df.to_csv("outputs/results_training.csv", mode='a', header=not file_exists, index=False)


In [ ]:
pd.DataFrame(results).to_pickle("outputs/results.pkl")
# neskôr: results_df = pd.read_pickle("results.pkl")

In [ ]:
print(cls_results.dtypes)

In [ ]:
# Výsledky klasifikácie
cls_results = new_df[new_df['target'] == 'RainTomorrow'][['model', 'features', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
print("=== KLASIFIKÁCIA (RainTomorrow) ===")
display(cls_results.style.highlight_max(subset=['f1', 'roc_auc'], color='lightgreen')
        .format('{:.4f}', subset=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'], na_rep='-'))

In [ ]:
# Výsledky regresie
reg_results = new_df[new_df['target'] == 'RISK_MM'][['model', 'features', 'MAE', 'RMSE', 'R2', 'derived_f1', 'derived_accuracy']]
print("=== REGRESIA (RISK_MM) ===")
display(reg_results.style
        .highlight_min(subset=['MAE', 'RMSE'], color='lightgreen')
        .highlight_max(subset=['R2', 'derived_f1'], color='lightgreen')
        .format('{:.4f}', subset=['MAE', 'RMSE', 'R2', 'derived_f1', 'derived_accuracy'], na_rep='-'))

## Zhrnutie Fázy 1

### Klasifikácia (RainTomorrow)

Najlepším klasifikačným modelom je `RandomForest` s najvyšším F1 (0.686) aj AUC (0.917) naprieč všetkými feature setmi. `LogisticRegression` dosahuje vysokú precision (0.72), ale nízky recall (0.47–0.50) — model je konzervatívny v predikcii dažďa. Toto by sa dalo zlepšiť úpravou klasifikačného thresholdu, keďže AUC je solídnych 0.87. `DecisionTree` je najslabší, ale stále rozumný baseline.

Rozdiely medzi feature setmi (`selected`, `no_critical`, `all`) sú malé, čo naznačuje, že ručne vybraný set dobre zachytáva podstatné informácie. Zaujímavé je porovnanie s originálnymi dátami — `DecisionTree` aj `LogisticRegression` na nespracovaných dátach dosiahli **vyššie** F1 než na transformovaných. To pravdepodobne súvisí s tým, že originálne dáta obsahujú `Location` (49 kategórií), ktorá je silnejšia featúra než `Region` (9 kategórií) použitý v transformovaných dátach.

### Regresia (RISK_MM)

Regresné modely predikujú množstvo zrážok v milimetroch. Najlepšie výsledky dosiahol `RandomForestRegressor` (R² = 0.40, MAE = 2.33 mm), nasledovaný `GradientBoosting` (R² = 0.38). `LinearRegression` slúži ako baseline a potvrdzuje, že vzťah medzi featúrami a množstvom zrážok nie je lineárny (R² = 0.21).

Celkovo je R² nízke — aj najlepší model vysvetľuje len 40 % variancie, čo je pri predikcii presného množstva zrážok očakávateľné. Zaujímavejšia je odvodená klasifikácia (predikcia ≥ 1 mm → dážď), kde `RandomForestRegressor` dosahuje F1 = 0.605, porovnateľne s priamym `DecisionTreeClassifier`.

Rozdiely medzi feature setmi sú opäť malé — `all` je mierne lepší, ale `selected` je veľmi blízko.

In [ ]:
# Vizualizácia: F1 porovnanie
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Klasifikácia — F1
pivot_f1 = cls_results.pivot(index='model', columns='features', values='f1')
pivot_f1.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('F1 score (klasifikácia)')
axes[0].set_ylim(0, 1)
axes[0].legend(title='Features')

# Klasifikácia — AUC
pivot_auc = cls_results.pivot(index='model', columns='features', values='roc_auc')
pivot_auc.plot(kind='bar', ax=axes[1], rot=0)
axes[1].set_title('ROC-AUC (klasifikácia)')
axes[1].set_ylim(0, 1)
axes[1].legend(title='Features')

plt.tight_layout()
plt.savefig('outputs/phase1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Fáza 2: Ladenie hyperparametrov

Na základe výsledkov Fázy 1 vyberieme najlepší model a feature set.

In [ ]:
# === Hyperparametre pre jednotlivé modely ===

param_grids = {
    'DecisionTree': {
        'model__max_depth': [5, 10, 15, 20, None],
        'model__min_samples_split': randint(2, 30),
        'model__min_samples_leaf': randint(1, 20),
        'model__criterion': ['gini', 'entropy'],
    },
    'LogisticRegression': {
        'model__C': uniform(0.01, 10),
        'model__penalty': ['l1', 'l2'],
        'model__solver': ['saga'],
    },
    'RandomForest': {
        'model__n_estimators': randint(100, 500),
        'model__max_depth': [10, 20, 30, None],
        'model__min_samples_split': randint(2, 20),
        'model__min_samples_leaf': randint(1, 10),
        'model__max_features': ['sqrt', 'log2', 0.3],
    },
}

In [ ]:
# === Tuning najlepšieho modelu ===
# Uprav best_model_name a best_fs_name podľa výsledkov Fázy 1

best_model_name = 'RandomForest'   # <- uprav podľa Fázy 1
best_fs_name = 'selected'          # <- uprav podľa Fázy 1

best_pipeline = clf_models[best_model_name]
best_features = feature_sets[best_fs_name]
X_best = df[best_features]

search = RandomizedSearchCV(
    best_pipeline,
    param_grids[best_model_name],
    n_iter=30,
    cv=cv_cls,
    scoring='f1',
    random_state=42,
    n_jobs=-1,
    verbose=1
)
search.fit(X_best, y_cls)

print(f"Best F1: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

In [ ]:
# Porovnanie: default vs tuned
default_f1 = cls_results.loc[(cls_results['model'] == best_model_name) & (cls_results['features'] == best_fs_name), 'f1'].values[0]
print(f"{best_model_name} ({best_fs_name}):")
print(f"  Default F1: {default_f1:.4f}")
print(f"  Tuned F1:   {search.best_score_:.4f}")
print(f"  Zlepšenie:  {search.best_score_ - default_f1:+.4f}")